# BAC testbed smoke test (Kaggle CPU)
Runs every testbed variant for a few steps in Kaggle's environment with full output. Everything heavy lives in /kaggle/tmp, so the saved output stays tiny.

In [ ]:
!python -c "import sys, torch, numpy, pandas; print(sys.version.split()[0], torch.__version__, numpy.__version__, pandas.__version__, 'cuda', torch.cuda.is_available())"
!df -h /kaggle/working /kaggle/tmp | cat; free -g

In [ ]:
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
!pip -q install transformers==4.36.2 tokenizers==0.15.2 timm==0.6.13 fvcore addict einops einops-exts==0.0.4 accelerate==0.21.0 peft==0.4.0 tabulate 2>&1 | tail -2

In [ ]:
import glob, os, shutil, zipfile
W = '/kaggle/tmp/Minor'; os.makedirs(W, exist_ok=True)
zips = glob.glob('/kaggle/input/**/bac_code.zip', recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(W)
else:
    marker = glob.glob('/kaggle/input/**/bac/common.py', recursive=True)
    assert marker, 'Add the bac-code dataset'
    shutil.copytree(os.path.dirname(os.path.dirname(marker[0])), W, dirs_exist_ok=True)
os.chdir(W)
!grep -c max-steps bac/token_bottleneck/testbed.py

In [ ]:
!python -c "from huggingface_hub import snapshot_download; snapshot_download('earth-insights/SegEarth-R1-EarthReason', local_dir='checkpoints/SegEarth-R1-EarthReason')" 2>&1 | tail -1
!python -c "from huggingface_hub import hf_hub_download; import zipfile; zipfile.ZipFile(hf_hub_download('earth-insights/EarthReason', 'val.zip', repo_type='dataset', local_dir='/kaggle/tmp/dl')).extractall('data/rs_reason_seg/RSReasonSeg')" 2>&1 | tail -1
!ls data/rs_reason_seg/RSReasonSeg/val/images | wc -l

In [ ]:
import subprocess
r = subprocess.run('python bac/token_bottleneck/extract_features.py --smoke 8', shell=True, capture_output=True, text=True)
print(r.returncode, r.stdout[-2000:], r.stderr[-4000:])
print(os.listdir('data/bottleneck_smoke'))

In [ ]:
tests = {
 'pool64': '--kind pool --tokens 64', 'uniform64': '--kind uniform --tokens 64', 'bac64': '--kind bac --tokens 64',
 'bac2_64': '--kind bac2 --tokens 64', 'bac2_64_randsel': '--kind bac2 --tokens 64 --select random',
 'bacq16': '--kind bacq --tokens 16', 'tome16': '--kind tome --tokens 16', 'random16': '--kind random --tokens 16',
 'pool1024': '--kind pool --tokens 1024', 'uniform32': '--kind uniform --tokens 32', 'bacq128': '--kind bacq --tokens 128'}
res = {}
for n, a in tests.items():
    r = subprocess.run(f'PY=python bash bac/token_bottleneck/run.sh {n} 1 0 {a} --feat-dir data/bottleneck_smoke --max-steps 3 --batch 4 --out-dir /kaggle/tmp/tb', shell=True, capture_output=True, text=True)
    ok = os.path.exists(f'/kaggle/tmp/tb/{n}_s0.json'); res[n] = ok
    print('=====', n, 'OK' if ok else 'FAILED'); print(r.stdout[-3000:], r.stderr[-2000:])
open('/kaggle/working/smoke_report.txt', 'w').write('\n'.join(f'{k} {v}' for k, v in res.items()))
print(res)

In [ ]:
!mkdir -p /kaggle/working/logs && cp -r results/logs/runs /kaggle/working/logs/ 2>/dev/null; ls /kaggle/working/logs/runs
assert all(res.values()), [k for k, v in res.items() if not v]